Для датафрейма log из материалов занятия создайте столбец source_type по правилам:
-если источник traffic_source равен Yandex или Google, то в source_type ставится organic;
-для источников paid и email из России ставим ad;
-для источников paid и email не из России ставим other;
-все остальные варианты берём из traffic_source без изменений.

In [1]:
import pandas as pd
import re

In [7]:
visit_log = pd.read_csv("vlog.csv", encoding="utf-8", sep=";")
visit_log["source_type"] = visit_log["traffic_source"]
visit_log.loc[visit_log["traffic_source"].isin(["yandex", "google"]), "source_type"] = "organic"
visit_log.loc[visit_log["traffic_source"].isin(["yandex", "google"]) & (visit_log["region"] == "Russia"), "source_type"] = "ad"
visit_log.loc[visit_log["traffic_source"].isin(["yandex", "google"]) & (visit_log["region"] != "Russia"), "source_type"] = "other"
visit_log.head()


,timestamp,visit_id,url,region,user_id,traffic_source,source_type
0,1549980692,e3b0c44298,https://host.ru/3c19b4ef7371864fa3,Russia,b1613cc09f,yandex,ad
1,1549980704,6e340b9cff,https://host.ru/c8d9213a31839f9a3a,Russia,4c3ec14bee,direct,direct
2,1549980715,96a296d224,https://host.ru/b8b58337d272ee7b15,Russia,a8c40697fb,yandex,ad
3,1549980725,709e80c884,https://host.ru/b8b58337d272ee7b15,Russia,521ac1d6a0,yandex,ad
4,1549980736,df3f619804,https://host.ru/b8b58337d272ee7b15,Russia,d7323c571c,yandex,ad


В файле URLs.txt содержатся URL страниц новостного сайта. Вам нужно отфильтровать его по адресам страниц с текстами новостей. Известно, что шаблон страницы новостей имеет внутри URL конструкцию: /, затем 8 цифр, затем дефис. Выполните действия:

-Прочитайте содержимое файла с датафрейм.
-Отфильтруйте страницы с текстом новостей, используя метод str.contains и регулярное выражение в соответствие с заданным шаблоном.

In [ ]:
urls = pd.read_csv("URLs.txt", encoding="utf-8")
urls_news = urls[urls['url'].str.contains(r'/\d{8}-', regex=True, na=False)].reset_index(drop=True)
urls_news.head()

,url
0,/politics/36188461-s-marta-zhizn-rossiyan-susc...
1,/world/36007585-tramp-pridumal-kak-reshit-ukra...
2,/science/36157853-nasa-sobiraet-ekstrennuyu-pr...
3,/video/36001498-poyavilis-pervye-podrobnosti-g...
4,/world/36007585-tramp-pridumal-kak-reshit-ukra...
...,...
79,/cis/35984145-kreml-prokommentiroval-soobschen...
80,/video/36071019-olimpiyskie-obekty-rio-prevrat...
81,/science/36151301-nazvano-posledstvie-zloupotr...
82,/incidents/36027330-vospitatelnitsu-zatravili-...


Используйте файл с оценками фильмов ml-latest-small/ratings.csv. Посчитайте среднее время жизни пользователей, которые выставили более 100 оценок. Под временем жизни понимается разница между максимальным и минимальным значениями столбца timestamp для данного значения userId.

In [ ]:
ratings = pd.read_csv("ratings.csv", encoding="utf-8")

answer = (
    ratings[ratings.groupby("userId")["rating"].transform("count") > 100]
    .groupby("userId")["timestamp"]
    .agg(lambda x: x.max() - x.min())
    .mean()
)

print(answer)


40080507.4496124


Дана статистика услуг перевозок клиентов компании по типам (см. файл “Python_13_join.ipynb” в разделе «Материалы для лекции “Продвинутый pandas”» ---- Ноутбуки к лекции «Продвинутый pandas»).
Нужно сформировать две таблицы:
-таблицу с тремя типами выручки для каждого client_id без указания адреса клиента;
-аналогичную таблицу по типам выручки с указанием адреса клиента.
Обратите внимание, что в процессе объединения таблиц данные не должны теряться. 

In [18]:
rzd = pd.DataFrame(
    {
        'client_id': [111, 112, 113, 114, 115],
        'rzd_revenue': [1093, 2810, 10283, 5774, 981]
    }
)
auto = pd.DataFrame(
    {
        'client_id': [113, 114, 115, 116, 117],
        'auto_revenue': [57483, 83, 912, 4834, 98]
    }
)
air = pd.DataFrame(
    {
        'client_id': [115, 116, 117, 118],
        'air_revenue': [81, 4, 13, 173]
    }
)
client_base = pd.DataFrame(
    {
        'client_id': [111, 112, 113, 114, 115, 116, 117, 118],
        'address': ['Комсомольская 4', 'Энтузиастов 8а', 'Левобережная 1а', 'Мира 14', 'ЗЖБИиДК 1',
                    'Строителей 18', 'Панфиловская 33', 'Мастеркова 4']
    }
)


In [28]:
all_revenue = rzd.merge(auto, on="client_id", how="outer").merge(air, on="client_id", how="outer")
all_revenue.fillna(0, inplace=True)


,client_id,rzd_revenue,auto_revenue,air_revenue
0,111,1093.0,0.0,0.0
1,112,2810.0,0.0,0.0
2,113,10283.0,57483.0,0.0
3,114,5774.0,83.0,0.0
4,115,981.0,912.0,81.0
5,116,0.0,4834.0,4.0
6,117,0.0,98.0,13.0
7,118,0.0,0.0,173.0


In [30]:
with_address = all_revenue.merge(client_base, on="client_id", how="outer")
with_address.fillna("нет данных", inplace=True)

,client_id,rzd_revenue,auto_revenue,air_revenue,address
0,111,1093.0,0.0,0.0,Комсомольская 4
1,112,2810.0,0.0,0.0,Энтузиастов 8а
2,113,10283.0,57483.0,0.0,Левобережная 1а
3,114,5774.0,83.0,0.0,Мира 14
4,115,981.0,912.0,81.0,ЗЖБИиДК 1
5,116,0.0,4834.0,4.0,Строителей 18
6,117,0.0,98.0,13.0,Панфиловская 33
7,118,0.0,0.0,173.0,Мастеркова 4
